# 📊 Анализ маркетинговых данных с Pandas и NumPy

В этом блокноте проводится исследовательский анализ данных (EDA), расчет бизнес-метрик (CTR, CR, CAC, ROAS, ROMI), сводные таблицы и визуализация прямо в Jupyter Notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка стилей для красивого отображения
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 20)

## 1. Загрузка данных и первичный осмотр
В Jupyter достаточно написать имя переменной (`df.head()`), чтобы получить красивую интерактивную HTML-таблицу с пагинацией.

In [ ]:
df = pd.read_csv('marketing_data.csv')
df['Күні'] = pd.to_datetime(df['Күні'], format='%d.%m.%Y')
df.head(10)

## 2. Расчет ключевых маркетинговых метрик (KPIs)

In [ ]:
# CTR: Кликабельность (%)
df['CTR_%'] = np.round((df['Клиент саны'] / df['Көрілім']) * 100, 2)

# CR: Конверсия из лида в заказ (%)
df['CR_%'] = np.round((df['Тапсырыс саны'] / df['Клиент саны']) * 100, 2)

# CAC: Стоимость привлечения одного заказа
df['CAC'] = np.round(df['Жарнама шығыны'] / df['Тапсырыс саны'], 2)

# AOV: Средний чек
df['AOV'] = np.round(df['Сатылым көлемі'] / df['Тапсырыс саны'], 2)

# ROAS: Возврат рекламных инвестиций (%)
df['ROAS_%'] = np.round((df['Сатылым көлемі'] / df['Жарнама шығыны']) * 100, 2)

# ROMI: Рентабельность маркетинговых инвестиций (%)
df['ROMI_%'] = np.round(((df['Пайда'] - df['Жарнама шығыны']) / df['Жарнама шығыны']) * 100, 2)

df[['Күні', 'Өнім атауы', 'Жарнама түрі', 'CTR_%', 'CR_%', 'CAC', 'ROAS_%', 'ROMI_%']].head()

## 3. Сравнение менеджеров со стилизацией (Styler)
В Jupyter можно использовать `style.background_gradient()`, чтобы подсвечивать значения цветом прямо в таблице (как тепловая карта)!

In [ ]:
manager_stats = df.groupby('Менеджер').agg(
    Тапсырыс_саны=('Тапсырыс саны', 'sum'),
    Сатылым_көлемі=('Сатылым көлемі', 'sum'),
    Таза_пайда=('Пайда', 'sum'),
    Орташа_чек=('AOV', 'mean')
).sort_values(by='Таза_пайда', ascending=False)

manager_stats.style.background_gradient(cmap='Blues').format({'Таза_пайда': '{:,.0f}', 'Орташа_чек': '{:.2f}'})

## 4. Сводная таблица (Pivot Table): Прибыль по Товарам и Городам

In [ ]:
pivot = pd.pivot_table(df, values='Пайда', index='Өнім атауы', columns='Қала', aggfunc='sum', fill_value=0)
pivot.style.background_gradient(cmap='YlGn')

## 5. Графики и визуализации в Jupyter
Все графики выводятся прямо под ячейками:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Прибыль по рекламным каналам
sns.barplot(data=df, x='Жарнама түрі', y='Пайда', estimator=sum, errorbar=None, palette='viridis', ax=axes[0])
axes[0].set_title('Общая прибыль по рекламным каналам', fontsize=13)
axes[0].set_ylabel('Суммарная прибыль')

# 2. Корреляционная матрица (Heatmap)
num_cols = ['Жарнама шығыны', 'Көрілім', 'Клиент саны', 'Тапсырыс саны', 'Сатылым көлемі', 'Пайда']
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1])
axes[1].set_title('Матрица корреляции Пирсона', fontsize=13)

plt.tight_layout()
plt.show()